In [ ]:
# Last amended: 27 Nov, 2026
# My folder:
# GitHUB repo: llamaindex/revised14042026/14.graph_rag
#


# 🎓 Lab Tutorial: Controlling Nodes & Relationships in GraphRAG (LlamaIndex + Ollama)

In `1.graphrag_student_tutorial.ipynb`, we used `SimpleLLMPathExtractor`, which lets the LLM freely decide what entities and relationships to extract from the documents. While convenient, free-form extraction often produces inconsistent labels (e.g., `CO_FOUNDED`, `FOUNDED_BY`, `CREATED` for the same concept) or noisy nodes.

In this tutorial, you will learn **two ways to take full control** over your Property Graph:

1. **Approach 1 — Schema-Guided Extraction (`SchemaLLMPathExtractor`)**: You define the allowed **entity types**, **relationship types**, and **valid `(Subject, Relation, Object)` schema rules**. The LLM still reads your documents automatically, but is strictly constrained to your ontology.
2. **Approach 2 — Manual / Exact Graph Construction (`EntityNode` & `Relation`)**: You explicitly define every single node instance, property, and edge in Python and upsert them directly into `SimplePropertyGraphStore` with zero LLM guesswork.

---

## 1. Setup & Environment Configuration
Just like in the first tutorial, we apply `nest_asyncio` for Jupyter compatibility, load the 4 sample documents from `data/`, and configure local **Ollama** models.

> **Tip:** For schema-constrained extraction, setting `temperature=0.0` (instead of `0.9`) helps local LLMs follow the JSON schema much more reliably.

In [8]:
%%time

# 1.0
# CRITICAL FOR JUPYTER NOTEBOOKS:
# Patch the running asyncio event loop so LlamaIndex can run smoothly.
import nest_asyncio
nest_asyncio.apply()

CPU times: user 65 μs, sys: 0 ns, total: 65 μs
Wall time: 68.4 μs


In [9]:
%%time

# 1.1
from pathlib import Path 
from llama_index.core import Settings, SimpleDirectoryReader, PropertyGraphIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# 1.2 Paths
#     Path class lets you treat file paths as smart objects
#        instead of plain text strings,
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"   # This is smart


CPU times: user 36 μs, sys: 6 μs, total: 42 μs
Wall time: 36.7 μs


In [10]:
# 1.3 Configure Local Ollama LLM (temperature=0.0 for deterministic schema extraction)
llm = Ollama(
    model="qwen2.5:latest",
    request_timeout=800.0,
    temperature=0.0,
)

# 1.4 Global Embedding Model
embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")

# 1.5
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

# 1.6 Load the 4 documents from data/
reader = SimpleDirectoryReader(input_dir=str(DATA_DIR))
documents = reader.load_data()
print(f"Loaded {len(documents)} documents from {DATA_DIR}")

Loaded 4 documents from /home/ashok/Documents/14.graph_rag/data


--- 
## 2. Approach 1: Schema-Guided Extraction (`SchemaLLMPathExtractor`)

Instead of `SimpleLLMPathExtractor`, we use **`SchemaLLMPathExtractor`**, which accepts four schema controls:

1. **`possible_entities`**: A `typing.Literal[...]` listing the exact node types allowed (e.g., `PERSON`, `COMPANY`, `INITIATIVE`, `TECHNOLOGY`, `LOCATION`).
2. **`possible_relations`**: A `typing.Literal[...]` listing the exact edge labels allowed (e.g., `CO_FOUNDED`, `INVESTED_IN`, `ADVISES`, `LICENSES_TO`).
3. **`kg_validation_schema`**: A list of allowed `(SubjectType, Relation, ObjectType)` triples (or a dict mapping each entity type to its allowed outgoing relations).
4. **`strict=True`**: Drops any extracted triplet that does not conform to your schema. *(Set `strict=False` if you want your schema to act as a soft hint rather than a hard filter.)*

**No external vector database** (such as Chroma, Qdrant, FAISS, or Pinecone) is used, **but an in-memory vector store (`SimpleVectorStore`) is used automatically under the hood in Approach 1.**

Here is how it breaks down in [2.graphrag_custom_schema_tutorial.ipynb](file:///home/ashok/Documents/14.graph_rag/2.graphrag_custom_schema_tutorial.ipynb):

1. **Approach 1 (`SchemaLLMPathExtractor` + `PropertyGraphIndex.from_documents`) — Yes (in-memory)**
   - In Steps 1.4–1.5 (the third setup code cell), we set `Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")`.
   - Because `SimplePropertyGraphStore` has no built-in vector index and `embed_kg_nodes=True` by default, `PropertyGraphIndex` automatically creates an **in-memory `SimpleVectorStore`** and embeds both the document chunks and the extracted graph nodes (that's why Step 2.5 logged `Generating embeddings: 0/4` and `Generating embeddings: 0/50`).
   - When you query `prop_index_schema.as_query_engine(include_text=True, similarity_top_k=3)`, LlamaIndex performs **hybrid retrieval**:
     - `VectorContextRetriever` searches the in-memory vector store (`similarity_top_k=3`) to find semantically relevant nodes and expands their graph neighbors.
     - `LLMSynonymRetriever` extracts keywords/synonyms from your question and matches them directly against node names in the graph.

2. **Approach 2 (Manual `EntityNode` + `Relation` + `PropertyGraphIndex.from_existing`) — No**
   - Because we upserted `EntityNode` and `Relation` objects directly into `SimplePropertyGraphStore` without embedding them or passing a `vector_store`, querying `index_manual` relies purely on **graph lookup/traversal** (`LLMSynonymRetriever`), with no vector search involved.

*(If you want to persist embeddings to a real vector database like Chroma or Qdrant in Approach 1, you can pass a `vector_store=...` argument to `PropertyGraphIndex.from_documents(...)`.)*

In [ ]:
%%time

# 2.0
from typing import Literal
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor

# 2.1 Define allowed Node Types (Entities)
entities = Literal[
    "PERSON",
    "COMPANY",
    "INITIATIVE",
    "TECHNOLOGY",
    "LOCATION",
    "REGULATOR",
]

# 2.2 Define allowed Relationship Types (Edges)
relations = Literal[
    "CO_FOUNDED",
    "WORKS_AT",
    "ADVISES",
    "SERVES_ON_BOARD_OF",
    "COLLABORATED_WITH",
    "INVESTED_IN",
    "FUNDED",
    "JOINT_VENTURE_WITH",
    "LICENSES_TO",
    "DEVELOPS",
    "TESTBED_FOR",
    "HEADQUARTERED_IN",
    "INVESTIGATES",
]

# 2.3 Define valid (SourceEntityType, Relation, TargetEntityType) rules
validation_schema = [
    ("PERSON", "CO_FOUNDED", "COMPANY"),
    ("PERSON", "WORKS_AT", "COMPANY"),
    ("PERSON", "ADVISES", "COMPANY"),
    ("PERSON", "SERVES_ON_BOARD_OF", "COMPANY"),
    ("PERSON", "COLLABORATED_WITH", "PERSON"),
    ("PERSON", "DEVELOPS", "TECHNOLOGY"),
    ("COMPANY", "INVESTED_IN", "COMPANY"),
    ("COMPANY", "FUNDED", "INITIATIVE"),
    ("COMPANY", "JOINT_VENTURE_WITH", "COMPANY"),
    ("COMPANY", "LICENSES_TO", "COMPANY"),
    ("COMPANY", "DEVELOPS", "TECHNOLOGY"),
    ("COMPANY", "TESTBED_FOR", "INITIATIVE"),
    ("COMPANY", "HEADQUARTERED_IN", "LOCATION"),
    ("REGULATOR", "INVESTIGATES", "PERSON"),
    ("REGULATOR", "INVESTIGATES", "COMPANY"),
]


In [12]:
%%time

# 2.4 Create the Schema-Constrained Extractor
kg_extractor = SchemaLLMPathExtractor(
                                    llm=llm,
                                    possible_entities=entities,
                                    possible_relations=relations,
                                    kg_validation_schema=validation_schema,
                                    strict=True,       # Enforce schema strictly (set False for soft guidance)
                                    max_triplets_per_chunk=10,
                                    num_workers=1,
                                    )

# 2.5
print("Constructing Schema-Guided Property Graph Index via Ollama...")
prop_index_schema = PropertyGraphIndex.from_documents(
                                                        documents,
                                                        kg_extractors=[kg_extractor],
                                                        use_async=False,
                                                        show_progress=True,
                                                    )
# 2.5.1
print("Schema-guided graph index built successfully!")

Constructing Schema-Guided Property Graph Index via Ollama...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-27 06:25:07,899 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-27 06:25:40,723 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-27 06:26:20,762 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-27 06:26:57,065 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

2026-11-27 06:26:59,144 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/50 [00:00<?, ?it/s]

2026-11-27 06:27:01,049 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-27 06:27:02,925 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-27 06:27:04,813 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-27 06:27:06,756 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-27 06:27:08,678 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Schema-guided graph index built successfully!
CPU times: user 60.6 ms, sys: 9.64 ms, total: 70.2 ms
Wall time: 2min 36s


### 2.1 Inspect the Extracted Triplets
Let's print out the extracted `(Subject [Label]) --[Relation]--> (Object [Label])` triples from the graph store to verify that every node and edge adheres to our schema.

In [16]:
%%time

# 3.0
from llama_index.core.graph_stores.types import EntityNode

# 3.1
# Note: SimplePropertyGraphStore.get_triplets() returns [] if called with no filters.
# Access `.graph.get_triplets()` directly to inspect all triplets in the store.
all_triplets = prop_index_schema.property_graph_store.graph.get_triplets()

# 3.2 Filter to Entity-to-Entity triplets (excluding document ChunkNode MENTIONS edges)
entity_triplets = [
                    (subj, rel, obj)
                    for subj, rel, obj in all_triplets
                    if isinstance(subj, EntityNode) and isinstance(obj, EntityNode)
                    ]



CPU times: user 71 μs, sys: 0 ns, total: 71 μs
Wall time: 77.5 μs


In [17]:
%%time

# 3.3
print(f"Total extracted entity triplets: {len(entity_triplets)} (out of {len(all_triplets)} total graph edges)\n")

# 3.4
for subj, rel, obj in entity_triplets:
    # 3.4.1
    print(f"({subj.name} [{subj.label}]) --[{rel.label}]--> ({obj.name} [{obj.label}])")

Total extracted entity triplets: 15 (out of 15 total graph edges)

(Aurora Technologies Inc. [COMPANY]) --[JOINT_VENTURE_WITH]--> (Helios Capital [COMPANY])
(Julian Mercer [PERSON]) --[SERVES_ON_BOARD_OF]--> (Aurora Technologies [COMPANY])
(Aurora Technologies Inc. [COMPANY]) --[HEADQUARTERED_IN]--> (Seattle, Washington [LOCATION])
(Helios Capital [COMPANY]) --[INVESTED_IN]--> (Synapse Dynamics [COMPANY])
(Aurora Technologies [COMPANY]) --[INVESTED_IN]--> (Synapse Dynamics [COMPANY])
(Dr. Elena Vance [PERSON]) --[WORKS_AT]--> (Aurora Technologies [COMPANY])
(Helios Capital [COMPANY]) --[INVESTED_IN]--> (Aether Labs [COMPANY])
(Synapse Dynamics [COMPANY]) --[DEVELOPS]--> (Synapse-OS cognitive architecture [TECHNOLOGY])
(Synapse Dynamics [COMPANY]) --[LICENSES_TO]--> (Aurora Technologies [COMPANY])
(Aurora Technologies Inc. [COMPANY]) --[LICENSES_TO]--> (Synapse Dynamics [COMPANY])
(Synapse Dynamics [COMPANY]) --[DEVELOPS]--> (Agile biomimetic humanoid robots [TECHNOLOGY])
(Helios Capita

### 2.2 Visualize the Schema-Guided Graph

In [18]:
%%time

# 4.0
from IPython.display import IFrame

# 4.1
prop_index_schema.property_graph_store.save_networkx_graph(name="./graph_schema.html")

# 4.2
IFrame(src="./graph_schema.html", width="100%", height="700px")

CPU times: user 12.9 ms, sys: 0 ns, total: 12.9 ms
Wall time: 12.2 ms


### 2.3 Query the Schema-Guided Graph

In [19]:
%%time

# 5.0
query_text = "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"

# 5.1
schema_query_engine = prop_index_schema.as_query_engine(include_text=True, similarity_top_k=3)

# 5.2
response_schema = schema_query_engine.query(query_text)

# 5.3
print("=== 💡 SCHEMA-GUIDED GRAPHRAG RESPONSE ===")
print(response_schema.response)

2026-11-27 06:29:07,559 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-27 06:29:07,586 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-27 06:29:26,016 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


=== 💡 SCHEMA-GUIDED GRAPHRAG RESPONSE ===
Dr. Elena Vance is connected to Synapse Dynamics and Project Nexus through Helios Capital in several ways:

1. **Intellectual Property and Licensing**: Dr. Elena Vance, as a co-founder of Aurora Technologies, brought proprietary CryoQubit compression research to the table. Aurora Technologies licenses optical communication protocols to Synapse Dynamics, an AI robotics company where Dr. Kenji Sato is the CEO. Although Dr. Vance recused herself from the Synapse Dynamics licensing vote due to undisclosed personal ties, the licensing agreement still exists.

2. **Project Nexus**: Helios Capital, which has invested in both Aurora Technologies and Synapse Dynamics, played a crucial role in stitching these entities together for Project Nexus. Julian Mercer, who has board positions at both Aurora Technologies and Aether Labs (a joint venture with Helios Capital), used his position to sanction undisclosed IP transfers, facilitating the integration of te

---
## 3. Approach 2: Manually Defining Exact Nodes & Relationships

What if you do **not** want the LLM to extract the graph at all — for example, you already have structured data (from a CSV, SQL database, JSON, or domain experts) and want **100% deterministic control** over every node, property, and edge?

You can build the graph directly using LlamaIndex's `EntityNode`, `Relation`, and `SimplePropertyGraphStore`, and then wrap it with `PropertyGraphIndex.from_existing(...)`.

In [25]:
%%time

# 6.0
# CRITICAL FOR JUPYTER NOTEBOOKS:
# Patch the running asyncio event loop so LlamaIndex can run smoothly.
import nest_asyncio
nest_asyncio.apply()

# 6.1
from pathlib import Path
from llama_index.core import Settings, SimpleDirectoryReader, PropertyGraphIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# 6.2 Paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

# 6.3 Configure Local Ollama LLM (temperature=0.0 for deterministic schema extraction)
llm = Ollama(
    model="qwen2.5:latest",
    request_timeout=800.0,
    temperature=0.0,
)

# 6.4 Global Embedding Model
embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")

# 6.5
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

# 6.6 Load the 4 documents from data/
reader = SimpleDirectoryReader(input_dir=str(DATA_DIR))
documents = reader.load_data()
print(f"Loaded {len(documents)} documents from {DATA_DIR}")

Loaded 4 documents from /home/ashok/Documents/14.graph_rag/data
CPU times: user 9.33 ms, sys: 2.83 ms, total: 12.2 ms
Wall time: 11.6 ms


In [26]:
%%time

# 7.0
from llama_index.core.graph_stores import (
    SimplePropertyGraphStore,
    EntityNode,
    Relation,
)

# 7.1 Create an empty in-memory Property Graph Store
manual_store = SimplePropertyGraphStore()

# 7.2 Explicitly define your Nodes (with labels and rich properties)
elena = EntityNode(
    name="Dr. Elena Vance",
    label="PERSON",
    properties={"role": "Co-Founder & Chief Scientific Advisor", "specialty": "CryoQubit & Quantum Compression"},
)
marcus = EntityNode(
    name="Marcus Thorne",
    label="PERSON",
    properties={"role": "Co-Founder & CEO"},
)
kenji = EntityNode(
    name="Dr. Kenji Sato",
    label="PERSON",
    properties={"role": "Founder & CEO", "background": "Former MIT robotics faculty"},
)
julian = EntityNode(
    name="Julian Mercer",
    label="PERSON",
    properties={"role": "Principal Investment Director"},
)
aurora = EntityNode(
    name="Aurora Technologies",
    label="COMPANY",
    properties={"headquarters": "Seattle, WA", "founded": 2021, "focus": "Quantum computing hardware"},
)
helios = EntityNode(
    name="Helios Capital",
    label="COMPANY",
    properties={"headquarters": "New York City", "fund": "Frontier Tech Fund IV ($1.2B)"},
)
aether = EntityNode(
    name="Aether Labs",
    label="COMPANY",
    properties={"type": "Joint Venture", "established": "September 2023"},
)
synapse = EntityNode(
    name="Synapse Dynamics",
    label="COMPANY",
    properties={"headquarters": "Boston", "founded": 2019, "focus": "Biomimetic humanoid robots"},
)
nexus = EntityNode(
    name="Project Nexus",
    label="INITIATIVE",
    properties={"status": "Stealth consortium", "goal": "Quantum-powered autonomous robotics"},
)

# 7.2.1
nodes = [elena, marcus, kenji, julian, aurora, helios, aether, synapse, nexus]

# 7.3 Explicitly define your Relationships between Nodes
relations = [
    Relation(label="CO_FOUNDED", source_id=elena.id, target_id=aurora.id, properties={"year": 2021}),
    Relation(label="CO_FOUNDED", source_id=marcus.id, target_id=aurora.id, properties={"year": 2021}),
    Relation(label="ADVISES", source_id=elena.id, target_id=synapse.id, properties={"since": "March 2024", "title": "Chief Scientific Advisor"}),
    Relation(label="COLLABORATED_WITH", source_id=elena.id, target_id=kenji.id, properties={"location": "Stanford University", "year": 2016}),
    Relation(label="CO_FOUNDED", source_id=kenji.id, target_id=synapse.id, properties={"year": 2019}),
    Relation(label="WORKS_AT", source_id=julian.id, target_id=helios.id),
    Relation(label="SERVES_ON_BOARD_OF", source_id=julian.id, target_id=aurora.id),
    Relation(label="SERVES_ON_BOARD_OF", source_id=julian.id, target_id=aether.id),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=aurora.id, properties={"amount": "$30M", "equity": "15%"}),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=synapse.id, properties={"amount": "$60M", "equity": "22%"}),
    Relation(label="JOINT_VENTURE_WITH", source_id=aurora.id, target_id=aether.id, properties={"partner": "Helios Capital"}),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=aether.id, properties={"amount": "$45M"}),
    Relation(label="LICENSES_TO", source_id=aurora.id, target_id=synapse.id, properties={"technology": "Optical Communication Protocols"}),
    Relation(label="FUNDED", source_id=helios.id, target_id=nexus.id, properties={"mode": "Covert"}),
    Relation(label="TESTBED_FOR", source_id=synapse.id, target_id=nexus.id),
]

# 7.4 Upsert Nodes FIRST, then Relations into the Graph Store
manual_store.upsert_nodes(nodes)
manual_store.upsert_relations(relations)

# 7.5 Wrap the populated store in a PropertyGraphIndex (no LLM extraction needed!)
# Note: use_async=False avoids nested-event-loop / sniffio errors in Jupyter
index_manual = PropertyGraphIndex.from_existing(
    property_graph_store=manual_store,
    llm=llm,
    embed_model=embed_model,
    use_async=False,
)

# 7.5.1
print(f"Manual graph created with {len(nodes)} nodes and {len(relations)} relationships!")

Manual graph created with 9 nodes and 15 relationships!
CPU times: user 1.12 ms, sys: 0 ns, total: 1.12 ms
Wall time: 1.07 ms


### 3.1 Visualize & Query the Manually Built Graph

In [27]:
%%time

# 8.0
from IPython.display import IFrame

# 8.1
index_manual.property_graph_store.save_networkx_graph(name="./graph_schema.html")

# 8.2
IFrame(src="./graph_schema.html", width="100%", height="700px")

CPU times: user 9.3 ms, sys: 757 μs, total: 10.1 ms
Wall time: 9.88 ms


In [28]:
%%time

# 9.0 Export and render the manual graph
# 9.1
index_manual.property_graph_store.save_networkx_graph(name="./graph_manual.html")

# 9.2
IFrame(src="./graph_manual.html", width="100%", height="650px")

CPU times: user 17 ms, sys: 0 ns, total: 17 ms
Wall time: 16.4 ms


In [29]:
%%time

# 10.0 Query the manually constructed graph
# Note: use_async=False ensures synchronous retrieval (avoids sniffio/asyncio errors in Jupyter)
manual_query_engine = index_manual.as_query_engine(include_text=False, use_async=False)

# 10.1
response_manual = manual_query_engine.query(
    "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"
)

# 10.2
print("=== 💡 MANUAL GRAPHRAG RESPONSE ===")
print(response_manual.response)

2026-11-27 06:42:08,120 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-11-27 06:42:12,698 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-27 06:42:12,721 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-27 06:42:20,295 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


=== 💡 MANUAL GRAPHRAG RESPONSE ===
Dr. Elena Vance is connected to Synapse Dynamics through her co-founding of the company and her advisory role. She is also linked to Project Nexus indirectly, as Synapse Dynamics serves as the testbed for Project Nexus, and Helios Capital, which invested in Synapse Dynamics, also funded Project Nexus.
CPU times: user 14.6 ms, sys: 2.17 ms, total: 16.8 ms
Wall time: 12.2 s


---
## 4. Quick Comparison Summary

| Method | Who Decides Node/Edge Types? | Who Decides Exact Instances? | Best Use Case |
| :--- | :--- | :--- | :--- |
| **`SimpleLLMPathExtractor`** | LLM | LLM | Quick exploration of unfamiliar text |
| **`SchemaLLMPathExtractor` (`strict=True`)** | **You** (`Literal` + `kg_validation_schema`) | LLM (constrained to your schema) | Production RAG over unstructured documents with a clean ontology |
| **`SchemaLLMPathExtractor` (`strict=False`)** | **You** (as a soft guide) + LLM | LLM | Semi-structured domains where new types may appear |
| **Manual `EntityNode` + `Relation`** | **You** | **You** | Structured data (CSV/DB/APIs) or curated gold-standard knowledge graphs |

In [ ]:
#############3